In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import roc_curve
from itertools import combinations
import pandas as pd
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

In [2]:
DEVICE     = "cpu"
NUM_CLASSES = 40
IMG_HEIGHT  = 112
IMG_WIDTH   = 112
ORL_FULL_PATH = "dataset/Training"   

SR_FULL_PATH_X2 = "dataset/Super_resolution/Training_SR_escala2"
SR_FULL_PATH_X4 = "dataset/Super_resolution/Training_SR_escala4"
CLAHE_FULL_X2    = "dataset/CLAHE/Training_CLAHE_escala2"
CLAHE_FULL_X4    = "dataset/CLAHE/Training_CLAHE_escala4"


In [3]:
full_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

def cargar_dataset_completo(path):

    dataset = datasets.ImageFolder(path, transform=full_transform)
    loader  = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)
    imagenes, etiquetas = [], []
    for img, label in loader:
        imagenes.append(img.squeeze(0))   # (1, H, W)
        etiquetas.append(label.item())
    return imagenes, etiquetas, dataset.classes

In [4]:
def extraer_embeddings(model, imagenes, es_arcface=False):
    """
    Extrae embeddings de todas las imágenes.
    - Modelos CE: corta antes del classifier, normaliza manualmente.
    - Modelos ArcFace: el forward() ya devuelve embedding L2-normalizado.
    """
    model.eval()
    embeddings = []

    with torch.no_grad():
        for img in imagenes:
            x = img.unsqueeze(0).to(DEVICE)   # (1, 1, H, W)

            if es_arcface:
                # ShuffleFaceNet2: forward() devuelve embedding L2-norm directamente
                emb = model(x)
            else:
                # ShuffleFaceNet (CE): cortar antes del classifier
                x = model.stem(x)
                x = model.stage1(x)
                x = model.stage2(x)
                x = model.stage3(x)
                x = model.stage4(x)
                x = model.conv_expand(x)
                x = model.gdc(x)
                emb = model.embedding(x)       # (1, 128)
                emb = F.normalize(emb, p=2, dim=1)

            embeddings.append(emb.squeeze(0).cpu().numpy())  # (128,)

    return np.array(embeddings)   # (N, 128)

In [5]:
def generar_pares(embeddings, etiquetas, n_impostores=10000, seed=42):
    """
    Genera pares genuinos (misma identidad) e impostores (distintas identidades).
    Retorna: similitudes coseno por par + etiqueta (1=genuino, 0=impostor)
    """
    np.random.seed(seed)
    etiquetas = np.array(etiquetas)
    n = len(embeddings)

    similitudes = []
    labels_pares = []

    # ── Pares genuinos: todas las combinaciones dentro de cada sujeto ──
    for sujeto in np.unique(etiquetas):
        indices = np.where(etiquetas == sujeto)[0]
        for i, j in combinations(indices, 2):
            sim = float(np.dot(embeddings[i], embeddings[j]))  # ya normalizados → coseno directo
            similitudes.append(sim)
            labels_pares.append(1)

    n_genuinos = len(similitudes)
    print(f"  Pares genuinos   : {n_genuinos}")

    # ── Pares impostores: muestreo aleatorio entre sujetos distintos ──
    impostores_generados = 0
    intentos = 0
    while impostores_generados < n_impostores and intentos < n_impostores * 10:
        i, j = np.random.randint(0, n, size=2)
        if etiquetas[i] != etiquetas[j]:
            sim = float(np.dot(embeddings[i], embeddings[j]))
            similitudes.append(sim)
            labels_pares.append(0)
            impostores_generados += 1
        intentos += 1

    print(f"  Pares impostores : {impostores_generados}")

    return np.array(similitudes), np.array(labels_pares)

In [6]:
def calcular_eer_tar(similitudes, labels_pares, far_objetivo=0.01):
    """
    Calcula EER y TAR@FAR dado un array de similitudes y etiquetas binarias.
    FAR  = False Acceptance Rate (impostores aceptados / total impostores)
    FRR  = False Rejection Rate  (genuinos rechazados / total genuinos)
    TAR  = True Acceptance Rate  = 1 - FRR
    EER  = punto donde FAR ≈ FRR
    """
    # roc_curve usa score alto = positivo → similitud alta = par genuino
    fpr, tpr, thresholds = roc_curve(labels_pares, similitudes)
    # fpr = FAR, tpr = TAR, fnr = FRR = 1 - TPR

    fnr = 1 - tpr

    # EER por interpolación: punto donde FAR cruza FRR
    eer_idx = np.argmin(np.abs(fpr - fnr))
    eer = float((fpr[eer_idx] + fnr[eer_idx]) / 2)

    # TAR @ FAR objetivo (ej: 1%)
    indices_far = np.where(fpr <= far_objetivo)[0]
    if len(indices_far) > 0:
        tar = float(tpr[indices_far[-1]])
    else:
        tar = 0.0

    return eer * 100, tar * 100   # en porcentaje

# Arquitectura base de shufflefacenet

In [7]:
class ConvBNPReLU(nn.Module):
    def __init__(self, in_c, out_c, kernel_size, stride=1, padding=0, groups=1):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, kernel_size, stride=stride,
                               padding=padding, groups=groups, bias=False)
        self.bn   = nn.BatchNorm2d(out_c)
        self.act  = nn.PReLU(out_c)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


def channel_shuffle(x, groups=2):
    b, c, h, w = x.size()
    channels_per_group = c // groups
    x = x.view(b, groups, channels_per_group, h, w)
    x = torch.transpose(x, 1, 2).contiguous()
    x = x.view(b, c, h, w)
    return x


class ShuffleDenseBlock(nn.Module):
    """Bloque tipo DenseNet + ShuffleNetV2 para etapas 2-4."""
    def __init__(self, in_c, stride=1):
        super().__init__()
        self.stride = stride
        branch_c = in_c // 2

        if stride == 1:
            self.branch1 = nn.Identity()
            branch1_out = branch_c
            branch2_in = branch_c
        else:
            self.branch1 = nn.Sequential(
                nn.Conv2d(in_c, in_c, 3, stride=stride, padding=1, groups=in_c, bias=False),
                nn.BatchNorm2d(in_c),
                nn.Conv2d(in_c, in_c, 1, bias=False),
                nn.BatchNorm2d(in_c),
                nn.PReLU(in_c)
            )
            branch1_out = in_c
            branch2_in = in_c

        mid_c = branch2_in * 2
        self.branch2 = nn.Sequential(
            ConvBNPReLU(branch2_in, mid_c, 1),
            nn.Conv2d(mid_c, mid_c, 3, stride=stride, padding=1, groups=mid_c, bias=False),
            nn.BatchNorm2d(mid_c),
            ConvBNPReLU(mid_c, branch2_in, 1)
        )

        self.out_channels = branch1_out + branch2_in

    def forward(self, x):
        if self.stride == 1:
            c = x.size(1) // 2
            x1, x2 = x[:, :c], x[:, c:]
            out = torch.cat([self.branch1(x1), self.branch2(x2)], dim=1)
        else:
            out = torch.cat([self.branch1(x), self.branch2(x)], dim=1)

        out = channel_shuffle(out, groups=2)
        return out


In [8]:
class ShuffleFaceNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, embedding_dim=128, dropout_p=0.4):
        super().__init__()

        # Stem: conv 3x3 stride=2 -> 112x112 -> 56x56
        # Reducido de 24 a 16 canales iniciales (modelo más pequeño para dataset chico)
        self.stem = ConvBNPReLU(1, 16, 3, stride=2, padding=1)

        # Etapa 1: 56x56 -> 28x28
        self.stage1 = nn.Sequential(ShuffleDenseBlock(16, stride=2))
        c1 = self.stage1[-1].out_channels  # 32

        # Etapa 2: 28x28 -> 14x14 (4 -> 3 bloques)
        stage2_blocks = [ShuffleDenseBlock(c1, stride=2)]
        c2 = stage2_blocks[-1].out_channels
        for _ in range(2):
            stage2_blocks.append(ShuffleDenseBlock(c2, stride=1))
            c2 = stage2_blocks[-1].out_channels
        self.stage2 = nn.Sequential(*stage2_blocks)

        # Etapa 3: 14x14 -> 7x7 (8 -> 4 bloques)
        stage3_blocks = [ShuffleDenseBlock(c2, stride=2)]
        c3 = stage3_blocks[-1].out_channels
        for _ in range(3):
            stage3_blocks.append(ShuffleDenseBlock(c3, stride=1))
            c3 = stage3_blocks[-1].out_channels
        self.stage3 = nn.Sequential(*stage3_blocks)

        # Etapa 4: 7x7 -> 7x7 (3 -> 2 bloques)
        stage4_blocks = []
        c4 = c3
        for _ in range(2):
            stage4_blocks.append(ShuffleDenseBlock(c4, stride=1))
            c4 = stage4_blocks[-1].out_channels
        self.stage4 = nn.Sequential(*stage4_blocks)

        # Conv 1x1 expansión - reducido de 512 a 256
        self.conv_expand = ConvBNPReLU(c4, 256, 1)

        # GDC: Global Depthwise Conv 7x7 -> reduce a (1,1)
        self.gdc = nn.Sequential(
            nn.Conv2d(256, 256, kernel_size=7, groups=256, bias=False),
            nn.GroupNorm(num_groups=16, num_channels=256)
        )

        # Embedding 128-D
        self.embedding = nn.Sequential(
            nn.Conv2d(256, embedding_dim, 1, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=embedding_dim),
            nn.Flatten()
        )

        # Dropout antes del classifier (regularización contra overfitting)
        self.dropout = nn.Dropout(p=dropout_p)

        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.conv_expand(x)
        x = self.gdc(x)
        x = self.embedding(x)
        x = self.dropout(x)
        return self.classifier(x)


model = ShuffleFaceNet(num_classes=NUM_CLASSES).to(DEVICE)
print(model)

dummy = torch.zeros(1, 1, IMG_HEIGHT, IMG_WIDTH).to(DEVICE)
out   = model(dummy)
print(f"\nShape de salida con input (1,1,112,112): {out.shape}")  # → (1, 40)

ShuffleFaceNet(
  (stem): ConvBNPReLU(
    (conv): Conv2d(1, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (act): PReLU(num_parameters=16)
  )
  (stage1): Sequential(
    (0): ShuffleDenseBlock(
      (branch1): Sequential(
        (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
        (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (3): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (4): PReLU(num_parameters=16)
      )
      (branch2): Sequential(
        (0): ConvBNPReLU(
          (conv): Conv2d(16, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, tra

# Shufflefacenet arquitectura ArcFace

In [9]:
class ConvBNPReLU2(nn.Module):
    def __init__(self, in_c, out_c, kernel_size, stride=1, padding=0, groups=1):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, kernel_size, stride=stride,
                               padding=padding, groups=groups, bias=False)
        self.bn   = nn.BatchNorm2d(out_c)
        self.act  = nn.PReLU(out_c)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


def channel_shuffle(x, groups=2):
    b, c, h, w = x.size()
    channels_per_group = c // groups
    x = x.view(b, groups, channels_per_group, h, w)
    x = torch.transpose(x, 1, 2).contiguous()
    x = x.view(b, c, h, w)
    return x

In [10]:
class ShuffleDenseBlock2(nn.Module):
    def __init__(self, in_c, stride=1):
        super().__init__()
        self.stride = stride
        branch_c = in_c // 2

        if stride == 1:
            self.branch1 = nn.Identity()
            branch1_out  = branch_c
            branch2_in   = branch_c
        else:
            self.branch1 = nn.Sequential(
                nn.Conv2d(in_c, in_c, 3, stride=stride, padding=1, groups=in_c, bias=False),
                nn.BatchNorm2d(in_c),
                nn.Conv2d(in_c, in_c, 1, bias=False),
                nn.BatchNorm2d(in_c),
                nn.PReLU(in_c)
            )
            branch1_out = in_c
            branch2_in  = in_c

        mid_c = branch2_in * 2
        self.branch2 = nn.Sequential(
            ConvBNPReLU2(branch2_in, mid_c, 1),
            nn.Conv2d(mid_c, mid_c, 3, stride=stride, padding=1, groups=mid_c, bias=False),
            nn.BatchNorm2d(mid_c),
            ConvBNPReLU2(mid_c, branch2_in, 1)
        )
        self.out_channels = branch1_out + branch2_in

    def forward(self, x):
        if self.stride == 1:
            c = x.size(1) // 2
            x1, x2 = x[:, :c], x[:, c:]
            out = torch.cat([self.branch1(x1), self.branch2(x2)], dim=1)
        else:
            out = torch.cat([self.branch1(x), self.branch2(x)], dim=1)
        return channel_shuffle(out, groups=2)

In [11]:
import math
import torch.nn.functional as F

class ArcFaceLoss2(nn.Module):
    def __init__(self, embedding_dim, num_classes, s=30.0, m=0.50):
        super().__init__()
        self.s           = s
        self.m           = m
        self.num_classes = num_classes
        # Matriz de pesos (un vector por clase), también normalizada
        self.weight      = nn.Parameter(torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_normal_(self.weight)          # Xavier Normal según tu análisis

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th    = math.cos(math.pi - m)           # umbral para estabilidad numérica
        self.mm    = math.sin(math.pi - m) * m

    def forward(self, embeddings, labels):
        # embeddings ya vienen normalizados desde el modelo (L2)
        # Normalizar también los pesos
        W = F.normalize(self.weight, dim=1)          # (num_classes, emb_dim)

        # coseno del ángulo entre embedding y cada vector de clase
        cos_theta  = F.linear(embeddings, W)         # (B, num_classes)
        cos_theta  = cos_theta.clamp(-1 + 1e-7, 1 - 1e-7)

        sin_theta  = torch.sqrt(1.0 - cos_theta ** 2)

        # cos(θ + m) = cos θ · cos m − sin θ · sin m
        cos_theta_m = cos_theta * self.cos_m - sin_theta * self.sin_m

        # Estabilidad: si θ > π − m, usar aproximación lineal
        cos_theta_m = torch.where(
            cos_theta > self.th,
            cos_theta_m,
            cos_theta - self.mm
        )

        # One-hot para seleccionar la clase correcta
        one_hot = torch.zeros_like(cos_theta)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)

        # Aplicar margen sólo a la clase correcta
        logits = one_hot * cos_theta_m + (1.0 - one_hot) * cos_theta
        logits = logits * self.s

        return F.cross_entropy(logits, labels)

In [12]:
class ShuffleFaceNet2_Arface(nn.Module):
    """
    Idéntica a ShuffleFaceNet pero:
      - Sin classifier lineal final
      - Embedding L2-normalizado (para ArcFace)
      - Sin Dropout (el margen angular ya regulariza)
      - Inicialización Xavier Normal en Conv y Linear
    """
    def __init__(self, embedding_dim=128):
        super().__init__()

        self.stem = ConvBNPReLU2(1, 16, 3, stride=2, padding=1)

        self.stage1 = nn.Sequential(ShuffleDenseBlock2(16, stride=2))
        c1 = self.stage1[-1].out_channels

        stage2 = [ShuffleDenseBlock2(c1, stride=2)]
        c2 = stage2[-1].out_channels
        for _ in range(2):
            stage2.append(ShuffleDenseBlock2(c2, stride=1))
            c2 = stage2[-1].out_channels
        self.stage2 = nn.Sequential(*stage2)

        stage3 = [ShuffleDenseBlock2(c2, stride=2)]
        c3 = stage3[-1].out_channels
        for _ in range(3):
            stage3.append(ShuffleDenseBlock2(c3, stride=1))
            c3 = stage3[-1].out_channels
        self.stage3 = nn.Sequential(*stage3)

        stage4 = []
        c4 = c3
        for _ in range(2):
            stage4.append(ShuffleDenseBlock2(c4, stride=1))
            c4 = stage4[-1].out_channels
        self.stage4 = nn.Sequential(*stage4)

        self.conv_expand = ConvBNPReLU2(c4, 256, 1)

        self.gdc = nn.Sequential(
            nn.Conv2d(256, 256, kernel_size=7, groups=256, bias=False),
            nn.GroupNorm(num_groups=16, num_channels=256)
        )

        # Proyección a embedding — SIN activación (lineal puro)
        self.embedding_proj = nn.Sequential(
            nn.Conv2d(256, embedding_dim, 1, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=embedding_dim),
            nn.Flatten()
        )

        # Inicialización Xavier Normal
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.conv_expand(x)
        x = self.gdc(x)
        x = self.embedding_proj(x)
        # L2-normalización: necesaria para ArcFace
        x = F.normalize(x, p=2, dim=1)
        return x   # shape: (B, 128) — embeddings normalizados
model = ShuffleFaceNet2_Arface(embedding_dim=128).to(DEVICE)
print(model)

dummy = torch.zeros(1, 1, IMG_HEIGHT, IMG_WIDTH).to(DEVICE)
out   = model(dummy)
print(f"\nShape de salida con input (1,1,112,112): {out.shape}")  # → (1, 128) embeddings normalizados

ShuffleFaceNet2_Arface(
  (stem): ConvBNPReLU2(
    (conv): Conv2d(1, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (act): PReLU(num_parameters=16)
  )
  (stage1): Sequential(
    (0): ShuffleDenseBlock2(
      (branch1): Sequential(
        (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
        (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (3): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (4): PReLU(num_parameters=16)
      )
      (branch2): Sequential(
        (0): ConvBNPReLU2(
          (conv): Conv2d(16, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bia

In [14]:
# Definir todos los checkpoints a evaluar
experimentos = [
    # (nombre, path_checkpoint, path_dataset, es_arcface)
    
    
    ("ORL original",                        "best_shufflefacenet_orl.pth",                    ORL_FULL_PATH, False),
    ("CLAHE x2 — CE",                       "best_shufflefacenet_clahe_x2_experimento2.pth",      CLAHE_FULL_X2, False),
    ("SR x2 — CE",                          "best_shufflefacenet_sr_x2_experimento2.pth",      SR_FULL_PATH_X2, False),
    ("CLAHE x4 — CE",                       "best_shufflefacenet_clahe_x4_experimento2.pth",      CLAHE_FULL_X4, False),
    ("SR x4 — CE",                          "best_shufflefacenet_sr_x4_experimento2.pth",      SR_FULL_PATH_X4, False),
    
    ("CLAHE x2 — ArcFace",                  "best_shufflefacenet2_clahe_x2_experimento3.pth",      CLAHE_FULL_X2, True),
    ("SR x2 — ArcFace",                     "best_shufflefacenet2_sr_x2_experimento3.pth",     SR_FULL_PATH_X2, True),
    ("CLAHE x4 — ArcFace",                  "best_shufflefacenet2_clahe_x4_experimento3.pth",      CLAHE_FULL_X4, True),
    ("SR x4 — ArcFace",                     "best_shufflefacenet2_sr_x4_experimento3.pth",     SR_FULL_PATH_X4, True),
    ("SR x4 — AdamW",                       "best_shufflefacenet_sr_x4_adamw.pth",             SR_FULL_PATH_X4, False),
]

resultados_eer_tar = []

for nombre, ckpt_path, dataset_path, es_arcface in experimentos:
    print(f"\n{'='*60}")
    print(f"Evaluando: {nombre}")
    print(f"  Checkpoint : {ckpt_path}")
    print(f"  Dataset    : {dataset_path}")
    print(f"  ArcFace    : {es_arcface}")

    # Cargar modelo
    if es_arcface:
        model = ShuffleFaceNet2_Arface(embedding_dim=128).to(DEVICE)
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model"] if "model" in ckpt else ckpt)
    else:
        model = ShuffleFaceNet(num_classes=NUM_CLASSES).to(DEVICE)
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))

    # Cargar dataset completo
    imagenes, etiquetas, clases = cargar_dataset_completo(dataset_path)
    print(f"  Imágenes cargadas: {len(imagenes)} ({len(clases)} sujetos)")

    # Extraer embeddings
    embeddings = extraer_embeddings(model, imagenes, es_arcface=es_arcface)
    print(f"  Embeddings shape : {embeddings.shape}")

    # Generar pares
    similitudes, labels_pares = generar_pares(embeddings, etiquetas, n_impostores=10000)

    # Calcular EER y TAR
    eer, tar_1 = calcular_eer_tar(similitudes, labels_pares, far_objetivo=0.01)
    _, tar_01  = calcular_eer_tar(similitudes, labels_pares, far_objetivo=0.001)

    print(f"  EER              : {eer:.2f}%")
    print(f"  TAR @ FAR=1%     : {tar_1:.2f}%")
    print(f"  TAR @ FAR=0.1%   : {tar_01:.2f}%")

    resultados_eer_tar.append({
        "Experimento": nombre,
        "Accuracy (Rank-1)": "ver tabla anterior",
        "EER (%)": round(eer, 2),
        "TAR @ FAR=1% (%)": round(tar_1, 2),
        "TAR @ FAR=0.1% (%)": round(tar_01, 2),
    })

df_resultados = pd.DataFrame(resultados_eer_tar)
print("\n" + "="*60)
print("TABLA RESUMEN — EER y TAR (ShuffleFaceNet SR)")
print("="*60)
print(df_resultados.to_string(index=False))


Evaluando: ORL original
  Checkpoint : best_shufflefacenet_orl.pth
  Dataset    : dataset/Training
  ArcFace    : False
  Imágenes cargadas: 360 (40 sujetos)
  Embeddings shape : (360, 128)
  Pares genuinos   : 1440
  Pares impostores : 10000
  EER              : 0.01%
  TAR @ FAR=1%     : 100.00%
  TAR @ FAR=0.1%   : 100.00%

Evaluando: CLAHE x2 — CE
  Checkpoint : best_shufflefacenet_clahe_x2_experimento2.pth
  Dataset    : dataset/CLAHE/Training_CLAHE_escala2
  ArcFace    : False
  Imágenes cargadas: 360 (40 sujetos)
  Embeddings shape : (360, 128)
  Pares genuinos   : 1440
  Pares impostores : 10000
  EER              : 2.64%
  TAR @ FAR=1%     : 92.99%
  TAR @ FAR=0.1%   : 81.94%

Evaluando: SR x2 — CE
  Checkpoint : best_shufflefacenet_sr_x2_experimento2.pth
  Dataset    : dataset/Super_resolution/Training_SR_escala2
  ArcFace    : False
  Imágenes cargadas: 360 (40 sujetos)
  Embeddings shape : (360, 128)
  Pares genuinos   : 1440
  Pares impostores : 10000
  EER              :

In [15]:
# Rank-1 obtenido en los experimentos de ShuffleFaceNet

rank1_conocido = {
    "ORL original":        100.00,

    # CrossEntropy (Experimento 2)
    "CLAHE x2 — CE":        97.50,
    "SR x2 — CE":           97.50,
    "CLAHE x4 — CE":        55.00,
    "SR x4 — CE":           82.50,

    # ArcFace (Experimento 3)
    "CLAHE x2 — ArcFace":   92.50,
    "SR x2 — ArcFace":      97.50,
    "CLAHE x4 — ArcFace":   85.00,
    "SR x4 — ArcFace":      95.00,

    # AdamW
    "SR x4 — AdamW":        92.50,
}

df_resultados["Rank-1 (%)"] = df_resultados["Experimento"].map(rank1_conocido)

df_final = df_resultados[
    [
        "Experimento",
        "Rank-1 (%)",
        "EER (%)",
        "TAR @ FAR=1% (%)",
        "TAR @ FAR=0.1% (%)"
    ]
]

print("TABLA FINAL — ShuffleFaceNet (Rank-1 + EER + TAR)")
print("=" * 80)
print(df_final.to_string(index=False))

# Guardar resultados
df_final.to_csv("resultados_eer_tar_shufflefacenet.csv", index=False)

print("\nGuardado en: resultados_eer_tar_shufflefacenet.csv")

TABLA FINAL — ShuffleFaceNet (Rank-1 + EER + TAR)
       Experimento  Rank-1 (%)  EER (%)  TAR @ FAR=1% (%)  TAR @ FAR=0.1% (%)
      ORL original       100.0     0.01            100.00              100.00
     CLAHE x2 — CE        97.5     2.64             92.99               81.94
        SR x2 — CE        97.5     0.46             99.86               97.64
     CLAHE x4 — CE        55.0     0.21            100.00               99.58
        SR x4 — CE        82.5     0.83             99.24               95.49
CLAHE x2 — ArcFace        92.5     0.01            100.00              100.00
   SR x2 — ArcFace        97.5     0.00            100.00              100.00
CLAHE x4 — ArcFace        85.0     0.00            100.00              100.00
   SR x4 — ArcFace        95.0     0.14            100.00               99.86
     SR x4 — AdamW        92.5     0.30             99.93               99.44

Guardado en: resultados_eer_tar_shufflefacenet.csv
